In [0]:
select distinct npi_num__v as hco_npi, corporate_name__v as hco_name_opendata
from com_edp_prd.com_raw.vod_hco
where npi_num__v in (
  '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364',
  '1093728743', '1184722779', '1861439952'
)

In [0]:
select distinct NPI as hco_npi, ORGANIZATION_NAME as hco_name
from com_edp_prd.com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952')

In [0]:
with base AS (
  SELECT DISTINCT
      a.npi_num__v        AS hco_npi,
      a.corporate_name__v AS hco_name,

      -- Hospital parent (lowest parent level in this rollup)
      b.npi_num__v        AS hospital_parent_npi,
      b.corporate_name__v AS hospital_parent_name,

      -- Immediate parent (mid-level)
      c.npi_num__v        AS immediate_parent_npi,
      c.corporate_name__v AS immediate_parent_name,

      -- Top parent (highest-level rollup)
      d.npi_num__v        AS top_parent_npi,
      d.corporate_name__v AS top_parent_name

  FROM com_raw.vod_hco a
  LEFT JOIN com_raw.vod_hco b
      ON a.hospital_parent__v = b.vid__v
  LEFT JOIN com_raw.vod_hco c
      ON a.immediate_parent__v = c.vid__v
  LEFT JOIN com_raw.vod_hco d
      ON a.top_parent__v = d.vid__v

  WHERE a.npi_num__v IN (
      '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952'
  )
),

/* ---------------------------------------------------------------------------
   STEP 2: Resolve a single parent NPI/name per HCO (best available parent)
   - Preference order:
       1) top parent
       2) immediate parent
       3) hospital parent
       4) null (no parent found)
   - Join the resolved parent fields back onto the hco_360 rows
   --------------------------------------------------------------------------- */
parent_mapping AS (
  SELECT DISTINCT
          hco_npi,
          hco_name,

          /* Parent NPI + Name resolved from SAME LEVEL */
          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_npi
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_npi
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_npi
              ELSE NULL
          END AS parent_npi,

          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_name
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_name
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_name
              ELSE NULL
          END AS parent_name
      FROM base
)

select * from parent_mapping

In [0]:
select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null)

In [0]:
with t1 as (select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null))
select a.npi_num__v as hco_npi, e.corporate_name__v as hco_name, a.top_parent__v, b.corporate_name__v as top_parent_name,
a.immediate_parent__v, c.corporate_name__v as immediate_parent_name,
a.hospital_parent__v, d.corporate_name__v as hospital_parent_name
from t1 as a
left join com_raw.vod_hco as b on a.top_parent__v = b.vid__v
left join com_raw.vod_hco as c on a.immediate_parent__v = c.vid__v
left join com_raw.vod_hco as d on a.hospital_parent__v = d.vid__v
left join com_raw.vod_hco as e on a.npi_num__v = e.npi_num__v

In [0]:
WITH t1 AS (
    SELECT DISTINCT 
        npi_num__v, 
        vid__v,
        top_parent__v, 
        immediate_parent__v, 
        hospital_parent__v 
    FROM com_raw.vod_hco 
    WHERE 
        top_parent__v = '931278329388861343'
        OR immediate_parent__v = '931278329388861343'
        OR hospital_parent__v = '931278329388861343'
)
SELECT 
    a.vid__v as child_vid,
    a.npi_num__v AS child_hco_npi,
    e.corporate_name__v AS child_hco_name,
    a.top_parent__v,
    b.corporate_name__v AS top_parent_name,
    a.immediate_parent__v,
    c.corporate_name__v AS immediate_parent_name,
    a.hospital_parent__v,
    d.corporate_name__v AS hospital_parent_name
FROM t1 AS a
LEFT JOIN com_raw.vod_hco AS b ON a.top_parent__v = b.vid__v
LEFT JOIN com_raw.vod_hco AS c ON a.immediate_parent__v = c.vid__v
LEFT JOIN com_raw.vod_hco AS d ON a.hospital_parent__v = d.vid__v
LEFT JOIN com_raw.vod_hco AS e ON a.npi_num__v = e.npi_num__v

### Parent Level Info (16th Jan)

In [0]:
WITH target_top_parents AS (
    SELECT DISTINCT top_parent__v
    FROM com_raw.vod_hco
    WHERE npi_num__v IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      AND top_parent__v IS NOT NULL
),

top_parent_child AS (
    SELECT
        tp.top_parent__v AS top_parent_vid,
        p.corporate_name__v AS top_parent_name,
        p.npi_num__v AS top_parent_npi,
        e.name AS top_parent_type,
        c.vid__v AS child_vid,
        c.corporate_name__v AS child_name,
        c.npi_num__v AS child_npi,
        d.name AS child_type,
        c.hco_status__v AS child_status
    FROM target_top_parents tp
    JOIN com_raw.vod_hco p
        ON tp.top_parent__v = p.vid__v
    JOIN com_raw.vod_hco c
        ON (tp.top_parent__v = c.top_parent__v
         OR tp.top_parent__v = c.immediate_parent__v
         OR tp.top_parent__v = c.hospital_parent__v)
         and c.npi_num__v in (select distinct hco_npi_old from com_edp_prd.cmpa_insights_internal_schema.reference_file)
    LEFT JOIN com_raw.vod_references d
        ON c.hco_type__v = d.code AND d.reference_type = 'HCOType'
    LEFT JOIN com_raw.vod_references e
        ON p.hco_type__v = e.code AND e.reference_type = 'HCOType'
    ORDER BY top_parent_vid, child_vid
),

appending_tier_flag AS (
  SELECT *,
    CASE
      WHEN child_npi IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      THEN 1 ELSE 0
    END AS tier1_flag
  FROM top_parent_child
),

child_count AS (
    SELECT top_parent_vid, COUNT(DISTINCT child_vid) AS no_child_accounts
    FROM appending_tier_flag
    GROUP BY top_parent_vid
),

appending_child_count AS (
  SELECT a.*, b.no_child_accounts
  FROM appending_tier_flag a
  LEFT JOIN child_count b
    ON a.top_parent_vid = b.top_parent_vid
),

/* -------------------------------------------------------------------------
   NEW: Tier1-only base_table built from your result (no hard-coded VALUES)
   ------------------------------------------------------------------------- */
base_table AS (
  SELECT DISTINCT TRY_CAST(child_npi AS STRING) AS hco_npi
  FROM appending_child_count
  WHERE tier1_flag = 1
    AND child_npi IS NOT NULL
),

hco_engaged AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
    AND a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_intgr.survey_target a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
),

tier1_flags AS (
  SELECT
    bt.hco_npi,
    CASE WHEN he.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_engaged,
    CASE WHEN hp.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_profiled
  FROM base_table bt
  LEFT JOIN hco_engaged he
    ON bt.hco_npi = he.npi
  LEFT JOIN hco_profiled hp
    ON bt.hco_npi = hp.npi
),

/* ------------------- your existing address/territory pipeline ------------------- */
target_vids AS (
    SELECT DISTINCT child_vid AS hco_vid
    FROM appending_child_count
    WHERE child_vid IS NOT NULL
    UNION
    SELECT DISTINCT top_parent_vid AS hco_vid
    FROM appending_child_count
    WHERE top_parent_vid IS NOT NULL
),

hco_address_latest AS (
    SELECT
        b.entity_vid__v AS hco_vid,
        b.address_line_1__v AS address_line_1,
        b.postal_code_cda__v AS postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY b.entity_vid__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_address b
    JOIN target_vids t
        ON t.hco_vid = b.entity_vid__v
    WHERE b.entity_type__v = 'HCO'
      AND b.record_state__v = 'VALID'
      AND b.address_status__v IN ('A','DS')
      AND b.address_verification_status__v NOT IN ('NS','U')
),

hco_address_latest_1 AS (
    SELECT hco_vid, address_line_1, postal_code
    FROM hco_address_latest
    WHERE rn = 1
),

addresses_child_top_parent AS (
    SELECT
        a.*,
        caddr.address_line_1 AS child_address,
        caddr.postal_code    AS child_zip,
        paddr.address_line_1 AS top_parent_address,
        paddr.postal_code    AS top_parent_zip
    FROM appending_child_count a
    LEFT JOIN hco_address_latest_1 caddr
        ON a.child_vid = caddr.hco_vid
    LEFT JOIN hco_address_latest_1 paddr
        ON a.top_parent_vid = paddr.hco_vid
),

territory_region_child AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS child_territory,
        COALESCE(z.region_name, '-')    AS child_region,
        COALESCE(z.city, '-')           AS child_city,
        COALESCE(z.state, '-')          AS child_state
    FROM addresses_child_top_parent a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.child_zip, '-') AS BIGINT) = z.zipcode
),

territory_region_parent AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS top_parent_territory,
        COALESCE(z.region_name, '-')    AS top_parent_region,
        COALESCE(z.city, '-')           AS top_parent_city,
        COALESCE(z.state, '-')          AS top_parent_state
    FROM territory_region_child a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.top_parent_zip, '-') AS BIGINT) = z.zipcode
),

/* -------------------------------------------------------------------------
   FINAL: attach engaged/profiled to CHILD NPI; NA when tier1_flag = 0
   ------------------------------------------------------------------------- */
final as (SELECT
  trp.*,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_engaged = 1 THEN '1'
    ELSE '0'
  END AS is_hco_engaged,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_profiled = 1 THEN '1'
    ELSE '0'
  END AS is_hco_profiled

FROM territory_region_parent trp
LEFT JOIN tier1_flags tf
  ON TRY_CAST(trp.child_npi AS STRING) = tf.hco_npi)

select * from final

In [0]:
WITH base_npis AS (
    select distinct hco_npi_old as npi
    from cmpa_insights_internal_schema.reference_file
),

hco_engaged AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_edp_prd.com_raw.vcrm_call2__v a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
    WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_intgr.survey_target a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
)

SELECT
    b.npi,
    CASE WHEN e.npi IS NOT NULL THEN 1 ELSE 0 END AS is_engaged,
    CASE WHEN p.npi IS NOT NULL THEN 1 ELSE 0 END AS is_profiled
FROM base_npis b
LEFT JOIN hco_engaged e
    ON b.npi = e.npi
LEFT JOIN hco_profiled p
    ON b.npi = p.npi
ORDER BY b.npi;


In [0]:
SELECT DISTINCT
    b.npi__v AS npi, a.entity_display_name__v as name_from_crm_calls_table
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  and a.call_date__v between '2025-07-01' and '2025-12-31'
  where b.npi__v is not null